In [ ]:
#here we have done model finetuning from the base model folder Deepscaler. the finetuned weights are in qlora_deepscaler_finetuned_2. 
# #the deepscaler_merged_model is the final model that contains both the mdoel and the weights updated . its based on feedback column in excel
pip install transformers datasets accelerate trl pandas torch


Note: you may need to restart the kernel to use updated packages.


In [1]:
#optional for kernal crashes avoidance
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import torch
print(torch.cuda.is_available())  # Should print True if GPU is detected
print(torch.cuda.device_count())  # Number of GPUs
print(torch.cuda.get_device_name(0))  # GPU name
torch.cuda.empty_cache()
print(torch.cuda.memory_summary())



True
1
NVIDIA GeForce RTX 4070 SUPER
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |  

In [3]:
import pandas as pd

# Replace 'your_file.xlsx' with the path to your Excel file
file_path = './RLCorrectedv4_with_detailed_feedback.xlsx'

# Load the Excel file into a DataFrame
df = pd.read_excel(file_path)

# Display the first few rows
print(df.head())


                                            Question  \
0  Write a function that returns the second small...   
1  Write a function that returns the second small...   
2  Write a function that returns the second small...   
3  Write a function that returns the second small...   
4  Write a function that returns the second small...   

                                     Reasoning_Chain                    Type  \
0  To find the second smallest number, we can fir...           Correct chain   
1  We can get the second smallest number by sorti...            Simple error   
2  The plan is to sort the list to find the secon...           Contradiction   
3  We want the second smallest number in the list...           Missing steps   
4  To find the second smallest number, one could ...  Irrelevant information   

   Reward_Score                                           Feedback  
0             5  Well done! The reasoning steps are logically s...  
1             4  Good attempt, but there's a

In [4]:
df['prompt'] = (
    "Question: " + df['Question'] +
    "\nReasoning Chain: " + df['Reasoning_Chain'] +
    "\n\nFeedback: "
)


In [5]:
#Convert to huggingface dataset
from datasets import Dataset

dataset = Dataset.from_pandas(df[['prompt', 'Type']])


In [6]:
dataset = dataset.train_test_split(test_size=0.1, seed=42)


In [7]:
from transformers import AutoTokenizer

model_name = "./Deepscaler"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Check and set pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Reuse eos_token as pad_token

def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["prompt"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["Type"],
            max_length=128,
            truncation=True,
            padding="max_length"
        )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/180 [00:00<?, ? examples/s]

c:\Users\zau3\anaconda3\Lib\site-packages\transformers\tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    llm_int8_skip_modules=None,
    offload_buffers=True  # This ensures offloading to CPU RAM
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


lora_config = LoraConfig(
    r=8,                      # Typical QLoRA rank
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Adjust based on model
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


Unused kwargs: ['offload_buffers']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qlora_deepscaler_finetuned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=5,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    push_to_hub=False
)


c:\Users\zau3\anaconda3\Lib\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [10]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator
)


C:\Users\zau3\AppData\Local\Temp\ipykernel_5788\76948488.py:8: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()


  0%|          | 0/110 [00:00<?, ?it/s]

{'loss': 2.5868, 'grad_norm': 0.6129407286643982, 'learning_rate': 0.00018181818181818183, 'epoch': 0.44}
{'loss': 2.2813, 'grad_norm': 0.8503827452659607, 'learning_rate': 0.00016363636363636366, 'epoch': 0.89}


  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 2.0820541381835938, 'eval_runtime': 0.7959, 'eval_samples_per_second': 25.129, 'eval_steps_per_second': 6.282, 'epoch': 1.0}
{'loss': 2.0528, 'grad_norm': 0.7754608988761902, 'learning_rate': 0.00014545454545454546, 'epoch': 1.31}
{'loss': 1.818, 'grad_norm': 0.7906151413917542, 'learning_rate': 0.00012727272727272728, 'epoch': 1.76}


  0%|          | 0/5 [00:00<?, ?it/s]

...

  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 1.5644292831420898, 'eval_runtime': 0.8043, 'eval_samples_per_second': 24.867, 'eval_steps_per_second': 6.217, 'epoch': 4.0}
{'loss': 1.4822, 'grad_norm': 0.7803193926811218, 'learning_rate': 1.8181818181818182e-05, 'epoch': 4.36}
{'loss': 1.5133, 'grad_norm': 0.7274977564811707, 'learning_rate': 0.0, 'epoch': 4.8}


  0%|          | 0/5 [00:00<?, ?it/s]

{'eval_loss': 1.5532073974609375, 'eval_runtime': 0.8002, 'eval_samples_per_second': 24.994, 'eval_steps_per_second': 6.249, 'epoch': 4.8}
{'train_runtime': 86.513, 'train_samples_per_second': 10.403, 'train_steps_per_second': 1.271, 'train_loss': 1.7995647257024592, 'epoch': 4.8}


TrainOutput(global_step=110, training_loss=1.7995647257024592, metrics={'train_runtime': 86.513, 'train_samples_per_second': 10.403, 'train_steps_per_second': 1.271, 'total_flos': 4100230710558720.0, 'train_loss': 1.7995647257024592, 'epoch': 4.8})

In [12]:
trainer.save_model("./qlora_deepscaler_finetuned_2")


In [13]:
#merge lora weights
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

# Paths
base_model_path = "./Deepscaler"
lora_weights_path = "./qlora_deepscaler_finetuned_2"

# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="auto"  # Use GPU if needed
)

# Load LoRA adapter
model = PeftModel.from_pretrained(model, lora_weights_path)


c:\Users\zau3\anaconda3\Lib\site-packages\accelerate\utils\modeling.py:1593: UserWarning: Current model requires 3712 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

c:\Users\zau3\anaconda3\Lib\site-packages\accelerate\utils\modeling.py:1593: UserWarning: Current model requires 7424 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


In [14]:
#merge lora weights
model = model.merge_and_unload()


In [15]:
#merge lora weights
output_path = "./deepscaler_merged_model"
model.save_pretrained(output_path)


In [6]:
#test the finetuned model with eval mode and parameter tuning
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model and tokenizer
model_path = "./deepscaler_merged_model"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set model to evaluation mode
model.eval()

# Prepare the prompt
prompt = (
    "you are a logic and facts expert, consider various aspects to be factual and give type of reasoning"
    "Question: Why is the sky blue?\n"
    "Reasoning Chain: Lets see, its because of scattering.\n"
    "Give very very concise Feedback with only one of the following options: \n"
    "1. Correct chain \n"
    "2. Simple error \n"
    "3. Contradiction \n"
    "4. Missing steps \n"
    "5. Irrelevant information"
)

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt")

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,       # Enable sampling
        no_repeat_ngram_size=2,
        top_k=5,              # Top-k sampling — limits to top 5 tokens
        top_p=0.9,            # Top-p nucleus sampling
        temperature=0.1       # Lower temp = less creative, more concise
    )

# Decode and print output
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Output:")
print(output_text)


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated Output:
you are a logic and facts expert, consider various aspects to be factual and give type of reasoningQuestion: Why is the sky blue?
Reasoning Chain: Lets see, its because of scattering.
Give very very concise Feedback with only one of the following options: 
1. Correct chain 
2. Simple error 
3. Contradiction 
4. Missing steps 
5. Irrelevant information
Feedback: 1
Reason: The reasoning chain is correct but lacks additional details about the scattering of light in the atmosphere.


In [2]:
#test the Base deepscaleR model with eval mode and parameter tuning
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model and tokenizer
model_path = "./Deepscaler"
model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set model to evaluation mode
model.eval()

# Prepare the prompt
prompt = (
    "you are a logic and facts expert, consider various aspects to be factual and give type of reasoning"
    "Question: Why is the sky blue?\n"
    "Reasoning Chain: Lets see, its because of scattering.\n"
    "Give very very concise Feedback with only one of the following options: \n"
    "1. Correct chain \n"
    "2. Simple error \n"
    "3. Contradiction \n"
    "4. Missing steps \n"
    "5. Irrelevant information"
)

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt")

# Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,       # Enable sampling
        no_repeat_ngram_size=2,
        top_k=5,              # Top-k sampling — limits to top 5 tokens
        top_p=0.9,            # Top-p nucleus sampling
        temperature=0.1       # Lower temp = less creative, more concise
    )

# Decode and print output
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated Output:")
print(output_text)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Generated Output:
you are a logic and facts expert, consider various aspects to be factual and give type of reasoningQuestion: Why is the sky blue?
Reasoning Chain: Lets see, its because of scattering.
Give very very concise Feedback with only one of the following options: 
1. Correct chain 
2. Simple error 
3. Contradiction 
4. Missing steps 
5. Irrelevant information
The options are:
A) Correct reasoning chain
B) Simple Error
C) Contr Ad
D) Missing Steps
E) Ir relevant
To choose the correct answer, I need to evaluate the reasoning provided and see if it aligns with the question.
The reasoning given is: "
